# Step 4. Benchmark question generation

In [1]:
import os
import pandas as pd
from typing import Any, Optional, Type
from pydantic import BaseModel
import pickle

from langchain_ollama import ChatOllama
from langchain_core.messages import BaseMessage
from langchain_core.prompts import PromptTemplate

import asyncio

In [2]:
SPLITS_PATH = './splits'
DATA_PATH = './data'
BASE_URL = 'http://192.168.1.43:11434/'  # 'http://localhost:11434/'
OLLAMA_MODEL = 'gemma3:27b-it-qat'  # 'gemma4:31b'  # 'qwen3:14b'
MIN_SPLIT_SIZE = 400

In [3]:
with open(os.path.join(SPLITS_PATH, 'splits.pkl'), 'rb') as f:
    splits = pickle.load(f)

In [4]:
splits[0].metadata

{'title': 'ЖУК В МУРАВЕЙНИКЕ',
 'id': '05fa57cd9e07c85d87b417cbe1fcd1c8',
 'size': 93,
 'collection': 'beetle_in_anthill'}

In [5]:
splits[2].metadata

{'title': 'ЖУК В МУРАВЕЙНИКЕ',
 'chapter': '1 июня 78–го года',
 'section': 'Я принял папку. С такой я...',
 'id': '9a490f7a02f67db0fa4bbb9710b1e9bb',
 'size': 1676,
 'collection': 'beetle_in_anthill'}

### Define and test the LLM

In [6]:
# OLLAMA_CONTEXT_LENGTH environment variable needs to be set to 16000
# set OLLAMA_CONTEXT_LENGTH=16000

In [7]:
llm = ChatOllama(
    model=OLLAMA_MODEL,
    temperature=0.5,
    base_url=BASE_URL,
    seed=42,
)

In [8]:
%%time
llm.invoke('Привет')

CPU times: total: 15.6 ms
Wall time: 2 s


AIMessage(content='Привет! Чем могу помочь?\n', additional_kwargs={}, response_metadata={'model': 'gemma3:27b-it-qat', 'created_at': '2026-04-14T08:41:02.0437549Z', 'done': True, 'done_reason': 'stop', 'total_duration': 1970920200, 'load_duration': 86915900, 'prompt_eval_count': 28, 'prompt_eval_duration': 430155400, 'eval_count': 14, 'eval_duration': 1449356300, 'logprobs': None, 'model_name': 'gemma3:27b-it-qat', 'model_provider': 'ollama'}, id='lc_run--019d8b26-a8b2-70c1-8d65-7ac5025cb955-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 28, 'output_tokens': 14, 'total_tokens': 42})

### Add structured output

In [9]:
class QuestionGenerationSchema(BaseModel):
    """Question Generation Schema for structured output"""
    number_of_questions: int
    questions: list[str]

In [10]:
def query_generation(system_prompt: str, md_text: str) -> QuestionGenerationSchema:
    content = f'\n\nФрагмент текста:\n\n{md_text}'
    try:
        messages = [
            {
                "role": "system",
                "content": system_prompt
            },
            {
                "role": "user",
                "content": content
            },
        ]
        structured_llm = llm.with_structured_output(QuestionGenerationSchema)
        response = structured_llm.invoke(messages)
        if not isinstance(response, QuestionGenerationSchema):
            raise RuntimeError("LLM did not return correct scheme response")
        return response
    except Exception as ex:
        print(f"Question generation error: {ex}")

### Prompt

In [11]:
QUESTION_GEN_PROMPT = '''
Твоя задача - придумать от {min_questions} до {max_questions} вопросов, опираясь на фрагмент художественного текста.
Формулируй вопросы так как если бы они задавались на викторине.
Убедись, что на вопросы можно ответить, опираясь на фрагмент текста.

В формулировке вопроса необходимо и очень **важно** использовать ключевые слова из текста:
- имена персонажей
- названия мест и организаций
- упомянутые в тексте предметы и гаджеты.

Правила подготовки вопросов:
- Вопросы должны быть заданы на русском языке.
- Вопросы не должны повторяться.
- В вопросе **нельзя** ссылаться на фрагмент текста,
**нельзя** использовать формулировки "согласно описанию в тексте", "согласно тексту", "как указано в тексте", "упоминается в фрагменте текста"

Вывод должен быть строго в формате JSON:
"number_of_questions": int, // Сколько вопросов ты подготовил для данного фрагмента текста.
"questions": [str, ..., str] // Список вопросов.
'''

In [12]:
prompt_template = PromptTemplate.from_template(QUESTION_GEN_PROMPT)
system_prompt = prompt_template.invoke({
    'min_questions': 1,
    'max_questions': 4,}).text

### Generate and save questions

In [13]:
%%time
result = pd.DataFrame()
for i in range(1, len(splits)):  # without toc chunk
    md_text = f'{splits[i].page_content}'
    metadata = splits[i].metadata
    
    size = metadata.get('size', 0)
    if size < MIN_SPLIT_SIZE:
        continue
    
    chapter_name = metadata.get('chapter')
    section_name = metadata.get('section')
    chunk_id = metadata.get('id')
   
    response = query_generation(system_prompt=system_prompt, md_text=md_text)
    print(f'Фрагмент: {i+1}')
    print(f'Глава: {chapter_name}')
    print(f'Секция: {section_name}')
    print(f'Кол-во вопросов: {response.number_of_questions}')
    for q in response.questions:
        print(f'- {q}')
        df_row = pd.DataFrame(data=[{
            'question': q,
            'chunk_id': chunk_id,
            'section_name': section_name,
            'chapter_name': chapter_name,
        }])
        result = pd.concat([result, df_row], ignore_index=True)
    print()

Фрагмент: 2
Глава: 1 июня 78–го года
Секция: СОТРУДНИК КОМКОНА–2 МАКСИМ КАММЕРЕР
Кол-во вопросов: 4
- Какое звание носит Максим Каммерер, к которому вызвал Экселенц?
- С какой организации, находящейся на Земле, отбыл Лев Вячеславович Абалкин незадолго до начала событий?
- Какой предмет извлек Экселенц из ящика стола, чтобы передать информацию о связях Абалкина?
- Какую инструкцию дал Экселенц Каммереру относительно взаимодействия с группой Каммерера в ходе расследования дела Абалкина?

Фрагмент: 3
Глава: 1 июня 78–го года
Секция: Я принял папку. С такой я...
Кол-во вопросов: 4
- На папке, переданной главному герою, было вытиснено имя и номер: «ЛЕВ ВЯЧЕСЛАВОВИЧ АБАЛКИН» «07». Что это за папка?
- Экселенц запретил главному герою использовать определенную технологию для изучения материалов. Как она называется?
- Экселенц упомянул планету, где Сикорски по прозвищу Странник преследовал Мак. Как называется эта планета?
- Сколько времени дано главному герою на выполнение задания, связанного с

In [14]:
print(f'Generated questions: {result.shape[0]}')

Generated questions: 378


In [15]:
result.head(10)

,question,chunk_id,section_name,chapter_name
0,"Какое звание носит Максим Каммерер, к которому...",037c58a8f1011d66e00e2488bd45380a,СОТРУДНИК КОМКОНА–2 МАКСИМ КАММЕРЕР,1 июня 78–го года
1,"С какой организации, находящейся на Земле, отб...",037c58a8f1011d66e00e2488bd45380a,СОТРУДНИК КОМКОНА–2 МАКСИМ КАММЕРЕР,1 июня 78–го года
2,"Какой предмет извлек Экселенц из ящика стола, ...",037c58a8f1011d66e00e2488bd45380a,СОТРУДНИК КОМКОНА–2 МАКСИМ КАММЕРЕР,1 июня 78–го года
3,Какую инструкцию дал Экселенц Каммереру относи...,037c58a8f1011d66e00e2488bd45380a,СОТРУДНИК КОМКОНА–2 МАКСИМ КАММЕРЕР,1 июня 78–го года
4,"На папке, переданной главному герою, было выти...",9a490f7a02f67db0fa4bbb9710b1e9bb,Я принял папку. С такой я...,1 июня 78–го года
5,Экселенц запретил главному герою использовать ...,9a490f7a02f67db0fa4bbb9710b1e9bb,Я принял папку. С такой я...,1 июня 78–го года
6,"Экселенц упомянул планету, где Сикорски по про...",9a490f7a02f67db0fa4bbb9710b1e9bb,Я принял папку. С такой я...,1 июня 78–го года
7,Сколько времени дано главному герою на выполне...,9a490f7a02f67db0fa4bbb9710b1e9bb,Я принял папку. С такой я...,1 июня 78–го года
8,Какую реакцию вызвали у Андрея и Сандро действ...,6d4f7864391ce37348f5806630431769,КОЕ–ЧТО О ЛЬВЕ АБАЛКИНЕ ПРОГРЕССОРЕ,1 июня 78–го года
9,Сколько сообщений накопилось на регистраторе з...,6d4f7864391ce37348f5806630431769,КОЕ–ЧТО О ЛЬВЕ АБАЛКИНЕ ПРОГРЕССОРЕ,1 июня 78–го года


In [16]:
result.to_csv(os.path.join(DATA_PATH, 'questions.csv'), sep=';', index=False)

In [17]:
min(result.question.str.len())

43